# LeetCode 2061 — Number of Spaces Cleaning Robot Cleaned bằng Python

Link đề: https://leetcode.com/problems/number-of-spaces-cleaning-robot-cleaned/ · Độ khó: **Medium** · Chủ đề: Array, Matrix, Simulation

> **Lưu ý: đây là bài Premium** trên LeetCode — cần tài khoản Premium để xem đề gốc và nộp bài. Đề dưới đây viết theo hiểu biết về bài (nguồn: bản đề của doocs/leetcode); nếu có tài khoản Premium, hãy **đối chiếu lại đề gốc**.

Skeleton LeetCode:

```python
class Solution:
    def numberOfCleanRooms(self, room: List[List[int]]) -> int:
        ...
```

## 1. Mục tiêu học

1. Mô phỏng một robot **tất định** (deterministic): trạng thái đầy đủ là `(hàng, cột, hướng)`, không chỉ vị trí.
2. Phát hiện **chu trình**: vì số trạng thái hữu hạn (`4mn`), robot chắc chắn lặp lại; dừng khi gặp lại một **trạng thái** đã thấy.
3. Hiểu vì sao dừng khi gặp lại **ô** đã thăm, hoặc chỉ khi về **ô xuất phát**, là sai.
4. Dùng **bitmask hướng** theo từng ô để lưu trạng thái gọn.
5. Viết mô phỏng **lặp** (không đệ quy) để an toàn với lưới `300 × 300`.

## 2. Đề bài & ví dụ

Căn phòng là ma trận nhị phân `room` (`0` trống, `1` có vật cản); ô trên-trái luôn trống. Robot bắt đầu ở ô trên-trái, **quay mặt sang phải**. Robot đi thẳng cho tới khi chạm **mép phòng** hoặc **vật cản** (ô kế tiếp), khi đó nó **quay 90° theo chiều kim đồng hồ** và lặp lại. Ô xuất phát và mọi ô robot đi qua đều được lau.

Trả về **số ô được lau** nếu robot chạy mãi mãi.

**Ví dụ 1:** `[[0,0,0],[1,1,0],[0,0,0]]` → `7`.

**Ví dụ 2:** `[[0,1,0],[1,0,0],[0,0,0]]` → `1` (robot bị kẹt, chỉ quay tại chỗ).

**Ví dụ 3:** `[[0,0,0],[0,0,0],[0,0,0]]` → `8` (ô giữa không bao giờ được lau).

**Ràng buộc:** `1 ≤ m, n ≤ 300`; `room[0][0] = 0`.

*(Đề Premium — đã diễn đạt lại, hãy đối chiếu lại đề gốc.)*

In [1]:
import random
import time
from typing import List

DIRS = [(0, 1), (1, 0), (0, -1), (-1, 0)]          # phải, xuống, trái, lên (theo chiều kim đồng hồ)
examples = [([[0, 0, 0], [1, 1, 0], [0, 0, 0]], 7), ([[0, 1, 0], [1, 0, 0], [0, 0, 0]], 1), ([[0, 0, 0], [0, 0, 0], [0, 0, 0]], 8)]
for room, exp in examples:
    print(room, "-> kỳ vọng", exp)

[[0, 0, 0], [1, 1, 0], [0, 0, 0]] -> kỳ vọng 7
[[0, 1, 0], [1, 0, 0], [0, 0, 0]] -> kỳ vọng 1
[[0, 0, 0], [0, 0, 0], [0, 0, 0]] -> kỳ vọng 8


## 3. Kiến thức nền

### 3.1 Hướng và quay theo chiều kim đồng hồ

Xếp các hướng theo thứ tự kim đồng hồ `phải → xuống → trái → lên`; quay phải = `d = (d + 1) % 4`.

In [2]:
names = ["phải", "xuống", "trái", "lên"]
d = 0
for _ in range(5):
    print(names[d], DIRS[d], end=" -> "); d = (d + 1) % 4
print()

phải (0, 1) -> xuống (1, 0) -> trái (0, -1) -> lên (-1, 0) -> phải (0, 1) -> 


### 3.2 Hệ tất định và chu trình

Một hệ mà trạng thái tiếp theo hoàn toàn xác định bởi trạng thái hiện tại, với hữu hạn trạng thái, sẽ **lặp lại** một trạng thái nào đó; từ đó nó đi vòng mãi (dạng "chữ ρ": một đoạn đuôi rồi một chu trình). Mọi ô được lau đều xuất hiện **trước** lần lặp trạng thái đầu tiên.

### 3.3 Trạng thái phải gồm cả hướng

Robot ở cùng một ô nhưng khác hướng sẽ đi tiếp **khác nhau**. Vì vậy "đã thăm ô này" **không** đồng nghĩa "sẽ lặp lại".

### 3.4 Bitmask hướng mỗi ô

Mỗi ô lưu một số 4 bit; bit `d` bật nghĩa là robot đã ở ô này với hướng `d`. Ô được lau ⇔ mask khác 0.

In [3]:
mask = 0
mask |= 1 << 0; mask |= 1 << 2
print(f"mask = {mask:04b}: đã ở ô với hướng {[names[d] for d in range(4) if mask >> d & 1]}")

mask = 0101: đã ở ô với hướng ['phải', 'trái']


## 4. Ý tưởng nền tảng / trực giác

Mô phỏng từng bước. Mỗi bước, từ trạng thái `(r, c, d)`:
- Nếu ô kế tiếp theo hướng `d` nằm trong phòng và trống ⇒ đi tới đó (giữ hướng).
- Ngược lại ⇒ quay phải tại chỗ (`d + 1`).

Dừng khi trạng thái `(r, c, d)` **đã gặp**. Đáp án = số ô khác nhau đã xuất hiện.

Tính tay **Ví dụ 2** `[[0,1,0],[1,0,0],[0,0,0]]`:

| bước | trạng thái | ô kế tiếp | hành động |
|---|---|---|---|
| 1 | (0,0, phải) | (0,1) vật cản | quay |
| 2 | (0,0, xuống) | (1,0) vật cản | quay |
| 3 | (0,0, trái) | ngoài phòng | quay |
| 4 | (0,0, lên) | ngoài phòng | quay |
| 5 | (0,0, phải) | — | **đã gặp** ⇒ dừng; 1 ô |

## 5. Cách 1 — Brute Force / baseline: chạy đủ số bước cố định

Vì chỉ có `4mn` trạng thái, sau `4mn` bước robot chắc chắn đã đi vào chu trình và mọi ô có thể lau đều đã lau. Chạy đúng `4mn + 1` bước rồi đếm — không cần phát hiện chu trình.

In [4]:
def clean_bruteforce(room):
    m, n = len(room), len(room[0])
    r = c = d = 0; cleaned = {(0, 0)}
    for _ in range(4 * m * n + 1):
        nr, nc = r + DIRS[d][0], c + DIRS[d][1]
        if 0 <= nr < m and 0 <= nc < n and room[nr][nc] == 0:
            r, c = nr, nc; cleaned.add((r, c))
        else:
            d = (d + 1) % 4
    return len(cleaned)

for room, exp in examples:
    got = clean_bruteforce(room); print(got, "kỳ vọng", exp); assert got == exp

7 kỳ vọng 7
1 kỳ vọng 1
8 kỳ vọng 8


**Độ phức tạp:** luôn `O(4mn)` bước (kể cả khi robot kẹt ngay từ đầu) — với `300 × 300` là `3.6·10⁵` bước, vẫn ổn. Nó đúng nhờ lập luận "đủ số bước", nhưng lãng phí khi chu trình ngắn.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (trạng thái):** `(r, c, d)` quyết định toàn bộ tương lai ⇒ gặp lại trạng thái là bắt đầu lặp.

**Insight 2 (dừng sớm):** dừng ngay lần đầu gặp lại trạng thái — số bước = độ dài đuôi + chu trình `≤ 4mn`.

**Insight 3 (lưu gọn):** mảng `mask[r][c]` 4 bit thay cho `set` các tuple — nhanh và ít bộ nhớ.

**Invariant:** tập ô đã lau tại mọi thời điểm = tập ô của các trạng thái đã đi qua.

### 6.1 Minh họa: đuôi và chu trình

Với phòng `[[0,0,0],[1,0,0]]` (ví dụ tự dựng, đối chiếu lại đề gốc), robot rời ô xuất phát rồi đi vào một chu trình **không** chứa ô xuất phát. In dãy trạng thái và vị trí bắt đầu chu trình.

In [5]:
def trajectory(room):
    m, n = len(room), len(room[0]); r = c = d = 0; seen = {}; seq = []
    while (r, c, d) not in seen:
        seen[(r, c, d)] = len(seq); seq.append((r, c, d))
        nr, nc = r + DIRS[d][0], c + DIRS[d][1]
        if 0 <= nr < m and 0 <= nc < n and room[nr][nc] == 0: r, c = nr, nc
        else: d = (d + 1) % 4
    return seq, seen[(r, c, d)]

seq, cyc = trajectory([[0, 0, 0], [1, 0, 0]])
print("dãy trạng thái:", [(r, c, names[d]) for r, c, d in seq])
print("chu trình bắt đầu ở bước", cyc, "-> ô xuất phát (0,0) chỉ nằm ở phần đuôi")

dãy trạng thái: [(0, 0, 'phải'), (0, 1, 'phải'), (0, 2, 'phải'), (0, 2, 'xuống'), (1, 2, 'xuống'), (1, 2, 'trái'), (1, 1, 'trái'), (1, 1, 'lên'), (0, 1, 'lên')]
chu trình bắt đầu ở bước 1 -> ô xuất phát (0,0) chỉ nằm ở phần đuôi


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mô phỏng + `set` các trạng thái

In [6]:
def clean_state_set(room):
    m, n = len(room), len(room[0])
    r = c = d = 0; seen = set(); cells = set()
    while (r, c, d) not in seen:
        seen.add((r, c, d)); cells.add((r, c))
        nr, nc = r + DIRS[d][0], c + DIRS[d][1]
        if 0 <= nr < m and 0 <= nc < n and room[nr][nc] == 0:
            r, c = nr, nc
        else:
            d = (d + 1) % 4
    return len(cells)

print([clean_state_set(r) for r, _ in examples])

[7, 1, 8]


**Độ phức tạp Cách 2:** `O(số trạng thái đi qua) ≤ O(4mn)` thời gian, cùng bậc bộ nhớ.

### 7.2 Cách 3 — Bitmask hướng mỗi ô (bản nộp)

In [7]:
class Solution:
    def numberOfCleanRooms(self, room: List[List[int]]) -> int:
        m, n = len(room), len(room[0])
        mask = [[0] * n for _ in range(m)]
        dr, dc = (0, 1, 0, -1), (1, 0, -1, 0)
        r = c = d = 0
        cleaned = 0
        while not mask[r][c] >> d & 1:
            if mask[r][c] == 0:
                cleaned += 1
            mask[r][c] |= 1 << d
            nr, nc = r + dr[d], c + dc[d]
            if 0 <= nr < m and 0 <= nc < n and room[nr][nc] == 0:
                r, c = nr, nc
            else:
                d = (d + 1) % 4
        return cleaned

print([Solution().numberOfCleanRooms(r) for r, _ in examples])

[7, 1, 8]


**Độ phức tạp Cách 3:** `O(4mn)` thời gian tệ nhất, `O(mn)` bộ nhớ (một số nhỏ mỗi ô).

### 7.3 Cách 4 — Đi theo "đoạn thẳng": nhảy cả đoạn rồi mới quay

Thay vì từng ô, robot đi thẳng tới khi bị chặn (vòng `while` bên trong), đánh dấu các ô dọc đường, rồi quay. Trạng thái kiểm tra lặp chỉ cần ở **điểm quay** `(r, c, d)` trước khi đi đoạn mới — ít lần kiểm tra hơn.

In [8]:
def clean_segments(room):
    m, n = len(room), len(room[0])
    seen_turn = set(); cleaned = {(0, 0)}; r = c = d = 0
    while (r, c, d) not in seen_turn:
        seen_turn.add((r, c, d))
        dr, dc = DIRS[d]
        while 0 <= r + dr < m and 0 <= c + dc < n and room[r + dr][c + dc] == 0:
            r += dr; c += dc; cleaned.add((r, c))
        d = (d + 1) % 4
    return len(cleaned)

print([clean_segments(r) for r, _ in examples])

[7, 1, 8]


Mỗi "đoạn" bắt đầu từ một trạng thái `(r, c, d)`; nếu trạng thái bắt đầu đoạn lặp lại, toàn bộ hành trình sau đó lặp lại. Lưu ý: ở điểm bắt đầu đoạn, robot có thể đi được 0 ô (bị chặn ngay) — vẫn hợp lệ.

## 8. TRACE từng bước

Trace Cách 3 trên **Ví dụ 1** `[[0,0,0],[1,1,0],[0,0,0]]`:

In [9]:
def trace(room, limit=40):
    m, n = len(room), len(room[0]); mask = [[0] * n for _ in range(m)]; r = c = d = 0; cleaned = 0; step = 0
    while not mask[r][c] >> d & 1 and step < limit:
        step += 1
        new = mask[r][c] == 0
        if new: cleaned += 1
        mask[r][c] |= 1 << d
        nr, nc = r + DIRS[d][0], c + DIRS[d][1]
        move = 0 <= nr < m and 0 <= nc < n and room[nr][nc] == 0
        print(f"bước {step:>2}: ({r},{c}) {names[d]:<5} {'ô mới' if new else '     '} -> {'đi tới (%d,%d)' % (nr, nc) if move else 'quay'}; đã lau {cleaned}")
        if move: r, c = nr, nc
        else: d = (d + 1) % 4
    print(f"dừng: trạng thái ({r},{c},{names[d]}) đã gặp")
    return cleaned

assert trace(examples[0][0]) == 7

bước  1: (0,0) phải  ô mới -> đi tới (0,1); đã lau 1
bước  2: (0,1) phải  ô mới -> đi tới (0,2); đã lau 2
bước  3: (0,2) phải  ô mới -> quay; đã lau 3
bước  4: (0,2) xuống       -> đi tới (1,2); đã lau 3
bước  5: (1,2) xuống ô mới -> đi tới (2,2); đã lau 4
bước  6: (2,2) xuống ô mới -> quay; đã lau 5
bước  7: (2,2) trái        -> đi tới (2,1); đã lau 5
bước  8: (2,1) trái  ô mới -> đi tới (2,0); đã lau 6
bước  9: (2,0) trái  ô mới -> quay; đã lau 7
bước 10: (2,0) lên         -> quay; đã lau 7
bước 11: (2,0) phải        -> đi tới (2,1); đã lau 7
bước 12: (2,1) phải        -> đi tới (2,2); đã lau 7
bước 13: (2,2) phải        -> quay; đã lau 7
dừng: trạng thái (2,2,xuống) đã gặp


Robot đi `(0,0) → (0,2)`, quay xuống, `(1,2) → (2,2)`, quay trái, `(2,1) → (2,0)`, quay lên (bị `1` chặn ở `(1,0)`), quay phải, đi `(2,1) → (2,2)` (đã lau), … cho tới khi gặp lại trạng thái `(2,2, phải)` hoặc tương tự. Tổng cộng 7 ô.

## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 3** — bitmask hướng mỗi ô.

In [10]:
class Solution:
    def numberOfCleanRooms(self, room: List[List[int]]) -> int:
        m, n = len(room), len(room[0])
        mask = [[0] * n for _ in range(m)]
        dr, dc = (0, 1, 0, -1), (1, 0, -1, 0)
        r = c = d = 0
        cleaned = 0
        while not mask[r][c] >> d & 1:
            if mask[r][c] == 0:
                cleaned += 1
            mask[r][c] |= 1 << d
            nr, nc = r + dr[d], c + dc[d]
            if 0 <= nr < m and 0 <= nc < n and room[nr][nc] == 0:
                r, c = nr, nc
            else:
                d = (d + 1) % 4
        return cleaned

for room, exp in examples:
    got = Solution().numberOfCleanRooms(room); print(got, exp); assert got == exp

7 7
1 1
8 8


## 10. Vì sao thuật toán đúng

Gọi `f` là hàm chuyển trạng thái `(r, c, d) ↦ (r', c', d')` theo luật của đề; dãy trạng thái `s₀ = (0,0,phải), s₁ = f(s₀), …` hoàn toàn xác định.

**Tồn tại lặp.** Có tối đa `4mn` trạng thái ⇒ tồn tại `i < j` với `sᵢ = sⱼ`. Gọi `j` nhỏ nhất như vậy. Vì `f` tất định, `s_{j+t} = s_{i+t}` với mọi `t ≥ 0` ⇒ mọi trạng thái về sau đều thuộc `{sᵢ, …, s_{j−1}}`.

**Đếm đúng.** Tập ô được lau "nếu chạy mãi" = các ô của `{s₀, …, s_{j−1}}`. Thuật toán duyệt đúng các trạng thái này và dừng ở `sⱼ` (trạng thái đầu tiên đã có trong mask). Một ô được đếm đúng một lần (khi mask của nó đang bằng 0). ∎

### 10.1 Kiểm chứng thực nghiệm

So bốn cách trên phòng ngẫu nhiên (mật độ vật cản khác nhau), và kiểm tra số bước mô phỏng `≤ 4mn`.

In [11]:
def rand_room(rng, m, n, p):
    g = [[1 if rng.random() < p else 0 for _ in range(n)] for _ in range(m)]; g[0][0] = 0
    return g

rng = random.Random(1010)
for _ in range(1500):
    m, n = rng.randint(1, 7), rng.randint(1, 7); room = rand_room(rng, m, n, rng.choice([0.1, 0.3, 0.5]))
    ref = clean_bruteforce(room)
    assert ref == clean_state_set(room) == clean_segments(room) == Solution().numberOfCleanRooms(room), room
    assert len(trajectory(room)[0]) <= 4 * m * n
print("4 cách khớp nhau trên 1500 phòng; số bước ≤ 4mn")

4 cách khớp nhau trên 1500 phòng; số bước ≤ 4mn


## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Chạy `4mn` bước | `Θ(4mn)` | `O(mn)` |
| 2. Set trạng thái | `O(số trạng thái đi qua)` | như trên |
| 3. Bitmask | `O(số trạng thái đi qua)` | `O(mn)` |
| 4. Theo đoạn | `O(số ô đi qua + số lần quay)` | `O(mn)` |

In [12]:
rng = random.Random(2061)
big_empty = [[0] * 300 for _ in range(300)]
big_rand = rand_room(rng, 300, 300, 0.02)
for name, f in [("brute", clean_bruteforce), ("state_set", clean_state_set), ("segments", clean_segments), ("submit", Solution().numberOfCleanRooms)]:
    t = time.perf_counter(); r1 = f(big_empty); r2 = f(big_rand); print(f"{name:<10}: {r1}, {r2} ({(time.perf_counter() - t) * 1000:.0f} ms)")

brute     : 1196, 243 (106 ms)
state_set : 1196, 243 (2 ms)
segments  : 1196, 243 (0 ms)
submit    : 1196, 243 (1 ms)


## 12. Bộ test

In [13]:
ALL = {"brute": clean_bruteforce, "state_set": clean_state_set, "segments": clean_segments, "submit": Solution().numberOfCleanRooms}
def run_all(room):
    res = {k: f(room) for k, f in ALL.items()}
    assert len(set(res.values())) == 1, (room, res)
    return res["submit"]

for room, exp in examples:
    assert run_all(room) == exp
# ví dụ tự dựng (đối chiếu lại đề gốc) — kiểm chéo bằng cách chạy đủ 4mn bước
edge = [([[0]], 1), ([[0, 0, 0]], 3), ([[0], [0], [0]], 3), ([[0, 1]], 1), ([[0, 0], [0, 0]], 4),
        ([[0, 0, 0], [1, 0, 0]], 5), ([[0, 0, 1], [1, 0, 0], [0, 0, 0]], 6)]
for room, exp in edge:
    assert run_all(room) == exp, (room, run_all(room))
rng = random.Random(20610)
for _ in range(500):
    run_all(rand_room(rng, rng.randint(1, 10), rng.randint(1, 10), rng.random() * 0.5))
for _ in range(3):
    run_all(rand_room(rng, 120, 120, 0.05))
print("Ví dụ, 7 edge case, 500 phòng nhỏ, 3 phòng 120x120: OK")

Ví dụ, 7 edge case, 500 phòng nhỏ, 3 phòng 120x120: OK


### 12.1 Kiểm tra tính chất

- Đáp án `≥ 1` và `≤` số ô trống.
- Phòng **toàn trống** `m × n` (với `m, n ≥ 2`): robot chỉ đi vòng viền ⇒ đáp án `2(m + n) − 4`.
- Thêm vật cản ở một ô trống **không kề** bất kỳ ô nào robot đi qua không đổi đáp án (mọi quyết định của robot chỉ phụ thuộc các ô kề với đường đi). Lưu ý: vật cản ở ô chưa lau nhưng **kề** đường đi có thể đổi điểm quay và đổi đáp án.

In [14]:
rng = random.Random(121)
f = Solution().numberOfCleanRooms
for m in range(2, 12):
    for n in range(2, 12):
        assert f([[0] * n for _ in range(m)]) == 2 * (m + n) - 4
for _ in range(500):
    m, n = rng.randint(1, 8), rng.randint(1, 8); room = rand_room(rng, m, n, 0.3)
    a = f(room); empty = sum(row.count(0) for row in room)
    assert 1 <= a <= empty
    cells = {(r, c) for r, c, _ in trajectory(room)[0]}
    far = [(i, j) for i in range(m) for j in range(n) if room[i][j] == 0 and all(abs(i - r) + abs(j - c) > 1 for r, c in cells)]
    if far:
        i, j = rng.choice(far); room2 = [row[:] for row in room]; room2[i][j] = 1
        assert f(room2) == a
print("Phòng trống: 2(m+n) − 4; 1 ≤ đáp án ≤ số ô trống; vật cản ở ô xa không ảnh hưởng")

Phòng trống: 2(m+n) − 4; 1 ≤ đáp án ≤ số ô trống; vật cản ở ô xa không ảnh hưởng


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dừng khi gặp lại một **ô** đã lau

In [15]:
def wrong_stop_on_cell(room):
    m, n = len(room), len(room[0]); cells = {(0, 0)}; r = c = d = 0; turns = 0
    while True:
        nr, nc = r + DIRS[d][0], c + DIRS[d][1]
        if 0 <= nr < m and 0 <= nc < n and room[nr][nc] == 0:
            if (nr, nc) in cells:
                return len(cells)                                  # SAI: cùng ô nhưng có thể khác hướng
            r, c = nr, nc; cells.add((r, c)); turns = 0
        else:
            d = (d + 1) % 4; turns += 1
            if turns == 4: return len(cells)

room = [[0, 0, 1], [1, 0, 0], [0, 0, 0]]        # tự dựng (đối chiếu lại đề gốc), tìm bằng thử ngẫu nhiên
print("sai :", wrong_stop_on_cell(room), "| đúng:", Solution().numberOfCleanRooms(room), "| brute:", clean_bruteforce(room))

sai : 5 | đúng: 6 | brute: 6


**Vì sao sai:** robot đi xuống tới `(2,1)`, rẽ trái sang `(2,0)`, bị chặn phía trên nên quay phải và đi **lại** qua `(2,1)` — lần này hướng sang phải và tiếp tục tới ô **chưa lau** `(2,2)`. Dừng ngay khi gặp lại ô `(2,1)` làm mất ô `(2,2)`. **Sửa:** dừng theo **trạng thái** `(r, c, d)`.

### Lỗi 2 — Chỉ dừng khi quay về **ô xuất phát**

In [16]:
def wrong_back_to_start(room, guard=10 ** 4):
    m, n = len(room), len(room[0]); cells = {(0, 0)}; r = c = d = 0; steps = 0
    while steps < guard:
        steps += 1
        nr, nc = r + DIRS[d][0], c + DIRS[d][1]
        if 0 <= nr < m and 0 <= nc < n and room[nr][nc] == 0:
            r, c = nr, nc; cells.add((r, c))
            if (r, c) == (0, 0): return len(cells), steps        # SAI: có thể không bao giờ quay lại
        else:
            d = (d + 1) % 4
    return "không dừng (vượt %d bước)" % guard, steps

room = [[0, 0, 0], [1, 0, 0]]                   # tự dựng (đối chiếu lại đề gốc)
print("sai :", wrong_back_to_start(room)[0], "| đúng:", Solution().numberOfCleanRooms(room))

sai : không dừng (vượt 10000 bước) | đúng: 5


**Vì sao sai:** chu trình có thể không chứa ô xuất phát (mục 6.1) ⇒ vòng lặp vô hạn. Ngay cả khi có quay lại, có thể quay lại với hướng khác và còn đi tiếp được. **Sửa:** phát hiện lặp trạng thái.

### Lỗi 3 — Quay rồi **đi luôn** trong cùng một bước (không kiểm tra hướng mới)

In [17]:
def wrong_turn_and_move(room):
    m, n = len(room), len(room[0]); seen = set(); cells = set(); r = c = d = 0
    while (r, c, d) not in seen:
        seen.add((r, c, d)); cells.add((r, c))
        nr, nc = r + DIRS[d][0], c + DIRS[d][1]
        if not (0 <= nr < m and 0 <= nc < n and room[nr][nc] == 0):
            d = (d + 1) % 4
            nr, nc = r + DIRS[d][0], c + DIRS[d][1]                 # SAI: không kiểm tra hướng mới có bị chặn
        if 0 <= nr < m and 0 <= nc < n:
            r, c = nr, nc
    return len(cells)

for room, exp in examples:
    print("sai :", wrong_turn_and_move(room), "| đúng:", exp)

sai : 8 | đúng: 7
sai : 8 | đúng: 1
sai : 8 | đúng: 8


**Vì sao sai:** sau khi quay, hướng mới cũng có thể bị chặn (Ví dụ 2: phải bị chặn, xuống cũng bị chặn). Bản sai "xuyên" qua vật cản. **Sửa:** quay là **một bước riêng**; bước sau mới kiểm tra lại.

### Lỗi 4 — Thứ tự hướng ngược chiều kim đồng hồ

In [18]:
def wrong_ccw(room):
    D2 = [(0, 1), (-1, 0), (0, -1), (1, 0)]                      # SAI: phải -> lên (ngược chiều)
    m, n = len(room), len(room[0]); seen = set(); cells = set(); r = c = d = 0
    while (r, c, d) not in seen:
        seen.add((r, c, d)); cells.add((r, c))
        nr, nc = r + D2[d][0], c + D2[d][1]
        if 0 <= nr < m and 0 <= nc < n and room[nr][nc] == 0: r, c = nr, nc
        else: d = (d + 1) % 4
    return len(cells)

for room, exp in examples:
    print("sai :", wrong_ccw(room), "| đúng:", exp)

sai : 3 | đúng: 7
sai : 1 | đúng: 1
sai : 8 | đúng: 8


**Vì sao sai:** đề: quay **theo** chiều kim đồng hồ: phải → **xuống** → trái → lên. **Sửa:** thứ tự `DIRS`.

### Lỗi 5 — Kiểm tra biên bằng chỉ số âm của Python

In [19]:
def wrong_negative_index(room):
    m, n = len(room), len(room[0]); seen = set(); cells = set(); r = c = d = 0
    while (r, c, d) not in seen and len(seen) < 4 * m * n + 5:
        seen.add((r, c, d)); cells.add((r, c))
        nr, nc = r + DIRS[d][0], c + DIRS[d][1]
        try:
            ok = nr < m and nc < n and room[nr][nc] == 0                 # SAI: thiếu nr >= 0, nc >= 0
        except IndexError:
            ok = False
        if ok: r, c = nr % m, nc % n
        else: d = (d + 1) % 4
    return len(cells)

room = [[0, 0, 0], [0, 1, 0], [0, 0, 0]]        # tự dựng (đối chiếu lại đề gốc)
print("sai :", wrong_negative_index(room), "| đúng:", Solution().numberOfCleanRooms(room))

sai : 7 | đúng: 8


**Vì sao sai:** `room[-1][c]` là hàng **cuối** — robot "xuyên mép" sang phía đối diện như phòng hình xuyến. **Sửa:** kiểm tra đủ `0 ≤ nr < m` và `0 ≤ nc < n`.

### Lỗi 6 — Đệ quy cho mô phỏng dài

In [20]:
import sys
def clean_recursive(room):
    m, n = len(room), len(room[0]); seen = set(); cells = set()
    def go(r, c, d):
        if (r, c, d) in seen: return
        seen.add((r, c, d)); cells.add((r, c))
        nr, nc = r + DIRS[d][0], c + DIRS[d][1]
        if 0 <= nr < m and 0 <= nc < n and room[nr][nc] == 0: go(nr, nc, d)
        else: go(r, c, (d + 1) % 4)
    go(0, 0, 0); return len(cells)

try:
    print(clean_recursive([[0] * 300 for _ in range(300)]))
except RecursionError as e:
    print("RecursionError trên phòng 300x300 trống:", str(e)[:40], "| bản lặp:", Solution().numberOfCleanRooms([[0] * 300 for _ in range(300)]))

RecursionError trên phòng 300x300 trống: maximum recursion depth exceeded | bản lặp: 1196


**Vì sao sai:** mỗi bước là một tầng đệ quy; hành trình dài hàng nghìn bước vượt giới hạn mặc định (~1000). **Sửa:** vòng lặp `while`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Mô phỏng hệ **tất định** chạy "mãi mãi" ⇒ tìm chu trình bằng cách lưu **trạng thái đầy đủ**.
- Trạng thái của robot = vị trí **+ hướng** (+ mọi biến khác ảnh hưởng tương lai).
- Hướng theo chiều kim đồng hồ: `(0,1), (1,0), (0,−1), (−1,0)`.
- Lưới lớn ⇒ vòng lặp, không đệ quy; bitmask nhỏ mỗi ô.
- Cận số bước = số trạng thái ⇒ luôn có lời giải "chạy đủ `4mn` bước" để kiểm chứng.

## 15. Biến thể của bài

1. **874 Walking Robot Simulation**, **1041 Robot Bounded In Circle**: robot theo lệnh, kiểm tra chu trình sau 4 vòng lệnh.
2. **2069 Walking Robot Simulation II**: robot đi vòng viền — đếm chu kỳ chu vi.
3. **Robot quay trái** khi bị chặn — đổi thứ tự hướng.
4. **Floyd/Brent** phát hiện chu trình với `O(1)` bộ nhớ (không cần mask) — Bài tập 5.
5. **Nhiều robot** cùng lúc — trạng thái là bộ vị trí–hướng của tất cả.

### 15.1 Code biến thể: robot quay **trái** khi bị chặn, kiểm chứng bằng chạy đủ bước

In [21]:
def clean_left_turn(room):
    m, n = len(room), len(room[0]); mask = [[0] * n for _ in range(m)]; r = c = 0; d = 0; cnt = 0
    while not mask[r][c] >> d & 1:
        if mask[r][c] == 0: cnt += 1
        mask[r][c] |= 1 << d
        nr, nc = r + DIRS[d][0], c + DIRS[d][1]
        if 0 <= nr < m and 0 <= nc < n and room[nr][nc] == 0: r, c = nr, nc
        else: d = (d - 1) % 4
    return cnt

def left_brute(room):
    m, n = len(room), len(room[0]); r = c = d = 0; cells = {(0, 0)}
    for _ in range(4 * m * n + 1):
        nr, nc = r + DIRS[d][0], c + DIRS[d][1]
        if 0 <= nr < m and 0 <= nc < n and room[nr][nc] == 0: r, c = nr, nc; cells.add((r, c))
        else: d = (d - 1) % 4
    return len(cells)

rng = random.Random(15)
for _ in range(500):
    room = rand_room(rng, rng.randint(1, 7), rng.randint(1, 7), 0.3); assert clean_left_turn(room) == left_brute(room)
print("Quay trái: bitmask = chạy đủ bước (500 phòng);", clean_left_turn([[0, 0, 0], [0, 0, 0]]))

Quay trái: bitmask = chạy đủ bước (500 phòng); 6


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Chạy `4mn` bước | Cận số trạng thái | `Θ(mn)` | `O(mn)` | Kiểm chứng |
| 2. Set trạng thái | Phát hiện lặp | `O(mn)` | `O(mn)` | Dễ hiểu |
| 3. Bitmask | Lặp + mask 4 bit | `O(mn)` | `O(mn)` | **Bản nộp** |
| 4. Theo đoạn | Kiểm tra lặp ở điểm quay | `O(mn)` | `O(mn)` | Ít kiểm tra |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `[[0]]`; (b) `[[0,0,0]]`; (c) `[[0,1]]`; (d) `[[0,0],[0,0]]`.

**Bài 2 (tính tay):** Trace phòng `[[0,0,1],[1,0,0],[0,0,0]]` và chỉ ra ô được lau **sau** lần thứ hai robot đi qua `(2,1)`.

**Bài 3 (điền TODO):** Hoàn thiện `step(room, r, c, d)` trả trạng thái kế tiếp.

**Bài 4 (tự viết lại):** Viết `cycle_info(room)` trả `(độ dài đuôi, độ dài chu trình)` của dãy trạng thái.

**Bài 5 (mini challenge):** Viết `clean_floyd(room)`: dùng thuật toán Floyd (rùa và thỏ) trên hàm `step` để tìm `μ` (điểm bắt đầu chu trình) và `λ` (độ dài chu trình) với `O(1)` bộ nhớ phụ cho việc phát hiện, rồi chạy lại `μ + λ` bước để đếm ô. Đối chiếu bản nộp.

In [22]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def step(room, r, c, d):
    m, n = len(room), len(room[0])
    # TODO: nếu ô kế tiếp theo DIRS[d] hợp lệ và trống thì trả (nr, nc, d); ngược lại (r, c, (d + 1) % 4)
    pass

In [23]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def cycle_info(room):
    # TODO: dùng dict trạng thái -> chỉ số bước; khi gặp lại: đuôi = chỉ số cũ, chu trình = bước hiện tại - chỉ số cũ
    pass

def clean_floyd(room):
    # TODO: rùa đi 1 bước, thỏ 2 bước đến khi gặp; tìm mu, lam; chạy lại mu + lam bước và đếm ô
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: robot quay tại chỗ khi kẹt. Mức 2: đi hết hàng rồi quay đầu. Mức 3: (a) `1`; (b) `3`; (c) `1`; (d) `4`.

**Bài 2.** Mức 1: robot đi `(0,0) → (0,1)`, bị chặn bởi `(0,2)`, quay xuống `(1,1) → (2,1)`. Mức 2: rẽ trái tới `(2,0)`, bị chặn phía trên bởi `(1,0)`, quay phải. Mức 3: đi lại qua `(2,1)` (hướng phải) tới `(2,2)` — ô mới; sau đó robot quay đầu và lặp lại chu trình trên hàng cuối (trạng thái `(2,1, trái)` lặp lại) ⇒ tổng 6 ô.

**Bài 3.** Mức 1: tính `nr, nc`. Mức 2: kiểm tra trong lưới và trống. Mức 3: trả tuple.

**Bài 4.** Mức 1: `seen[state] = index`. Mức 2: khi lặp, `mu = seen[state]`. Mức 3: `lam = index − mu`.

**Bài 5.** Mức 1: tìm điểm gặp bằng rùa/thỏ. Mức 2: đặt rùa về đầu, đi cùng tốc độ để tìm `μ`. Mức 3: từ điểm `μ`, đi tới khi quay lại để có `λ`.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2, 3
f = Solution().numberOfCleanRooms
b1 = [f(x) for x in ([[0]], [[0, 0, 0]], [[0, 1]], [[0, 0], [0, 0]])]
print("Bài 1:", b1); assert b1 == [1, 3, 1, 4]
seq, cyc = trajectory([[0, 0, 1], [1, 0, 0], [0, 0, 0]])
print("Bài 2:", [(r, c, names[d]) for r, c, d in seq], "| chu trình từ bước", cyc)
def step(room, r, c, d):
    m, n = len(room), len(room[0]); nr, nc = r + DIRS[d][0], c + DIRS[d][1]
    if 0 <= nr < m and 0 <= nc < n and room[nr][nc] == 0:
        return nr, nc, d
    return r, c, (d + 1) % 4
assert step([[0, 0]], 0, 0, 0) == (0, 1, 0) and step([[0, 0]], 0, 1, 0) == (0, 1, 1)
print("Bài 3: OK")

Bài 1: [1, 3, 1, 4]
Bài 2: [(0, 0, 'phải'), (0, 1, 'phải'), (0, 1, 'xuống'), (1, 1, 'xuống'), (2, 1, 'xuống'), (2, 1, 'trái'), (2, 0, 'trái'), (2, 0, 'lên'), (2, 0, 'phải'), (2, 1, 'phải'), (2, 2, 'phải'), (2, 2, 'xuống'), (2, 2, 'trái')] | chu trình từ bước 5
Bài 3: OK


In [25]:
# Bài 4, 5
def cycle_info(room):
    seq, mu = trajectory(room)
    return mu, len(seq) - mu

def clean_floyd(room):
    s0 = (0, 0, 0)
    f_ = lambda s: step(room, *s)
    tort, hare = f_(s0), f_(f_(s0))
    while tort != hare:
        tort, hare = f_(tort), f_(f_(hare))
    mu, tort = 0, s0
    while tort != hare:
        tort, hare = f_(tort), f_(hare); mu += 1
    lam, hare = 1, f_(tort)
    while tort != hare:
        hare = f_(hare); lam += 1
    s, cells = s0, set()
    for _ in range(mu + lam):
        cells.add(s[:2]); s = f_(s)
    return len(cells)

rng = random.Random(5)
for _ in range(500):
    room = rand_room(rng, rng.randint(1, 8), rng.randint(1, 8), 0.3)
    mu, lam = cycle_info(room)
    assert clean_floyd(room) == f(room)
    seq, _ = trajectory(room); assert mu + lam == len(seq)
print("Bài 4, 5: OK |", cycle_info([[0, 0, 0], [1, 0, 0]]), clean_floyd(examples[0][0]))

Bài 4, 5: OK | (1, 8) 7


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
mask = [[0] * n for _ in range(m)]
r = c = d = 0; cleaned = 0
while not mask[r][c] >> d & 1:            # trạng thái (r, c, d) chưa gặp
    if mask[r][c] == 0: cleaned += 1
    mask[r][c] |= 1 << d
    nr, nc = r + dr[d], c + dc[d]
    if 0 <= nr < m and 0 <= nc < n and room[nr][nc] == 0: r, c = nr, nc
    else: d = (d + 1) % 4                  # quay là MỘT bước riêng
return cleaned
```

**Checklist:**
- [ ] Phát hiện lặp theo **trạng thái** `(r, c, d)`, không theo ô?
- [ ] Không giả định robot quay về ô xuất phát?
- [ ] Quay là một bước riêng (kiểm tra lại hướng mới)?
- [ ] Thứ tự hướng **theo** chiều kim đồng hồ?
- [ ] Kiểm tra biên đủ hai phía (không dùng chỉ số âm)?